In [1]:
# STEP 1 — Imports and setup

import os
import pandas as pd

BASE_DIR = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

building_ids = [
    d for d in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, d))
]

print(f"Found {len(building_ids)} building folders")

Found 117 building folders


In [2]:
# STEP 2 — Create the standard 30-minute timestep vector (17,520 rows)
# Starts at 00:30:00 (NOT 00:00:00)

time_index = pd.date_range(
    start="00:30:00",
    periods=17520,
    freq="30min"
).time

# Convert to required string format HH:MM:SS
time_strings = [t.strftime("%H:%M:%S") for t in time_index]

print("Generated time vector:", len(time_strings))

Generated time vector: 17520


In [3]:
# STEP 3 — Process a single building (NO MERGE, PURE COLUMN OPERATION)

def process_building(building_id):

    try:
        appliance_path = os.path.join(
            BASE_DIR, building_id,
            "APPLIANCES",
            "appliance_power_w_lighting.csv"
        )

        lighting_path = os.path.join(
            BASE_DIR, building_id,
            "LIGHTING",
            "lighting_power.csv"
        )

        output_path = os.path.join(
            BASE_DIR, building_id,
            "APPLIANCES",
            "appliance_power.csv"
        )

        # Load data
        appliance_df = pd.read_csv(appliance_path)
        lighting_df = pd.read_csv(lighting_path)

        # Extract numeric columns only (ignore Time entirely for safety)
        appliance_vals = appliance_df.iloc[:, 1].values
        lighting_vals = lighting_df.iloc[:, 1].values

        # Safety check
        if len(appliance_vals) != 17520 or len(lighting_vals) != 17520:
            raise ValueError(
                f"Unexpected length in {building_id}: "
                f"{len(appliance_vals)} / {len(lighting_vals)}"
            )

        # Compute new appliance power
        appliance_new = appliance_vals - lighting_vals

        # Build output dataframe
        output_df = pd.DataFrame({
            "Time": time_strings,
            "appliance_power_new": appliance_new
        })

        # Save (overwrite if exists)
        output_df.to_csv(output_path, index=False)

        print(f"Processed {building_id}")

    except Exception as e:
        print(f"Error in {building_id}: {e}")

In [4]:
# STEP 4 — Run for all buildings

for building_id in building_ids:
    process_building(building_id)

print("All buildings completed.")

Processed 1001664924
Processed 1001829085
Processed 1001063639
Processed 1001829071
Processed 1234761002
Processed 1002539407
Processed 1001063637
Processed 1001664941
Processed 1001991633
Processed 1235057414
Processed 1001829079
Processed 1001664922
Processed 1234761003
Processed 1001664925
Processed 1000238907
Processed 1234761004
Processed 1002313096
Processed 1001870878
Processed 1001664940
Processed 1234982990
Processed 1234806523
Processed 1001870876
Processed 1001870882
Processed 1002143357
Processed 1001951902
Processed 1234621926
Processed 1234647955
Processed 1001906269
Processed 1001951903
Processed 1001627539
Processed 1002143351
Processed 1001627541
Processed 1236594950
Processed 1001627570
Processed 1002031280
Processed 1001627549
Processed 1001627540
Processed 1001627547
Processed 1001870854
Processed 1001430744
Processed 1234760995
Processed 1235812262
Processed 1000336709
Processed 1001991628
Processed 1001664938
Processed 1000238925
Processed 1002143291
Processed 100